In [11]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = 'BmYsInLDmrI'

# 1. Cria a instância da API
youtube_api = YouTubeTranscriptApi()

try:
    # 2. Lista as transcrições disponíveis
    transcript_list = youtube_api.list(video_id)
    
    # 3. Encontra a transcrição (preferencialmente em português)
    transcript = transcript_list.find_transcript(['pt', 'en'])
    
    # 4. Busca os dados
    # O .fetch() retorna uma lista de dicionários padrão
    texto_dados = transcript.fetch()
    
    # 5. Exibe o resultado
    for item in texto_dados:
        # Se for dicionário, item['text'] funciona. 
        # Se for objeto, item.text funcionará.
        texto = item.get('text') if isinstance(item, dict) else item.text
        tempo = item.get('start') if isinstance(item, dict) else item.start
        print(f"{tempo}: {texto}")

except Exception as e:
    print(f"Erro ao processar: {e}")

0.28: Esse é o novo Ford Territory
6.24: 2025. Mudou tudo, agora anda de verdade.
10.16: É um dos maiores espaços internos da
13.08: categoria. Mas será que tem preço e
15.92: condições para bater de frente com os
18.84: outros chineses?
23.32: Não, você não me escutou errado, não.
26.72: Esse, apesar de ter o emblema da Ford,
28.96: vem da China. O SUV de entrada agora da
32.32: Ford no Brasil, mas que não é feito pela
35.399: Ford. E além disso, vale a pena dizer
37.84: que esse carro lá fora já ganhou o
41.0: facelift e virou híbrido. Aqui também
43.96: vai ser igual no vídeo de hoje a gente
45.52: vai ver tudo sobre essa máquina aqui.
47.52: Vamos falar da motorização, vamos falar
50.16: do câmbio do espaço interno que ele dá
52.32: um banho nos outros da categoria do
54.52: porta presuntos. E é claro que a gente
55.96: vai levar essa máquina, dar um rolê e
57.359: ver qual é que é. E é claro que não tem
59.28: lugar melhor pra gente começar a falar
61.359: do Forge Territory, do q

In [ ]:
from googleapiclient.discovery import build
import os
from dotenv import load_dotenv
# Carrega as variáveis do arquivo .env para o ambiente do sistema
load_dotenv()

# Acessa a variável
api_key = os.getenv('YOUTUBE_API_KEY')
video_id = 'BmYsInLDmrI'

youtube = build('youtube', 'v3', developerKey=api_key)

# Chamada para buscar comentários
response = youtube.commentThreads().list(
    part='snippet',
    videoId=video_id,
    maxResults=100,  # Máximo de 100 por requisição
    textFormat='plainText'
    order='relevance'
).execute()

response['items'][0]
for item in response['items']:
    comentario = item['snippet']['topLevelComment']['snippet']['textDisplay']
    like_count = item['snippet']['topLevelComment']['snippet']['likeCount']
    rep_total = item['snippet']['totalReplyCount']
    resp = item['replies']
    autor = item['snippet']['topLevelComment']['snippet']['authorDisplayName']
    print(f"{autor}: {comentario}")

{'kind': 'youtube#commentThread',
 'etag': 'ktprSd2lJNiWDOOB5_QSKnatQvg',
 'id': 'UgxQDmlacTl-O0lJZYF4AaABAg',
 'snippet': {'channelId': 'UCMk1wBIsHmalYeE16GkD6bg',
  'videoId': 'BmYsInLDmrI',
  'topLevelComment': {'kind': 'youtube#comment',
   'etag': 'ud37k45b5Cjzd2m_CRdl8Ysgo_s',
   'id': 'UgxQDmlacTl-O0lJZYF4AaABAg',
   'snippet': {'channelId': 'UCMk1wBIsHmalYeE16GkD6bg',
    'videoId': 'BmYsInLDmrI',
    'textDisplay': 'Tenho um 2025. Carro sensacional. Pelo valor e por tudo que oferece, não vejo uma opção confiável que seja melhor.',
    'textOriginal': 'Tenho um 2025. Carro sensacional. Pelo valor e por tudo que oferece, não vejo uma opção confiável que seja melhor.',
    'authorDisplayName': '@viniciusvasconcelos8398',
    'authorProfileImageUrl': 'https://yt3.ggpht.com/ytc/AIdro_kvPzSWHTfu9lxV3HpVDEHbJAkPRxjo2fEfeXTmGg=s48-c-k-c0x00ffffff-no-rj',
    'authorChannelUrl': 'http://www.youtube.com/@viniciusvasconcelos8398',
    'authorChannelId': {'value': 'UC9EaWYlDUAethDuC2F1bPH

In [ ]:
from googleapiclient.discovery import build
import os
from dotenv import load_dotenv

load_dotenv()
youtube = build('youtube', 'v3', developerKey=os.getenv('YOUTUBE_API_KEY'))

def buscar_review_carro(marca, modelo, tipo,dt_ini,dt_fim):
    # Monta uma query de busca otimizada
    # Usar "review" ou "avaliação" ajuda a filtrar conteúdos opinativos
    query = f"review avaliação {marca} {modelo} {tipo}"
    
    request = youtube.search().list(
        q=query,
        part='snippet',
        type='video',
        maxResults=5,
        order='relevance',         # Prioriza os mais relevantes
        relevanceLanguage='pt',    # Foca em português   # Filtra por vídeos em HD (melhor UX para reviews)
        publishedAfter=dt_ini,
        publishedBefore=dt_fim
    )
    
    return request.execute()

from datetime import datetime, timedelta, timezone

def definir_periodo_busca(meses_atras=6):
    # Data atual em UTC no formato RFC 3339
    agora = datetime.now(timezone.utc)
    
    # Data limite (agora menos X meses)
    data_inicio = agora - timedelta(days=meses_atras * 30)
    
    # Formatação para RFC 3339 (ex: 2026-04-21T12:00:00Z)
    # O 'Z' indica UTC, que é o padrão exigido pelo YouTube
    published_after = data_inicio.strftime('%Y-%m-%dT%H:%M:%SZ')
    published_before = agora.strftime('%Y-%m-%dT%H:%M:%SZ')
    
    return published_after, published_before

# Obter as datas
inicio, fim = definir_periodo_busca(6)

# Exemplo de uso dinâmico:
resultado = buscar_review_carro("Toyota", "Corolla", "sedan")

for item in resultado['items']:
    video_id = item['id']['videoId']
    titulo = item['snippet']['title']
    descricao = item['snippet']['description']
    canal = item['snippet']['channelTitle']
    data_publicacao = item['snippet']['publishedAt']

    # Acessando a miniatura de alta qualidade
    thumbnail_url = item['snippet']['thumbnails']['high']['url']

    # Montando a URL do vídeo
    url_video = f"https://www.youtube.com/watch?v={video_id}"

    print(f"Título: {titulo}")
    print(f"Canal: {canal}")
    print(f"Data: {data_publicacao}")
    print(f"URL da Thumbnail: {thumbnail_url}")
    print(f"Link: {url_video}")

    print("=" * 80)

Título: O MELHOR COROLLA DE TODOS OS TEMPOS! Avaliação Toyota Corolla ALTIS 2014
Canal: Carro Chefe
Data: 2026-01-16T20:55:14Z
URL da Thumbnail: https://i.ytimg.com/vi/t7JqSehIbGE/hqdefault.jpg
Link: https://www.youtube.com/watch?v=t7JqSehIbGE
Título: Avaliação Toyota Corolla SEG 2005 Automático - o melhor Corolla já lançado!
Canal: Carro Chefe
Data: 2019-04-25T21:57:43Z
URL da Thumbnail: https://i.ytimg.com/vi/srvo2JouVno/hqdefault.jpg
Link: https://www.youtube.com/watch?v=srvo2JouVno
Título: Avaliação Toyota Corolla Hibrido 2025 - IMPRESSIONA MUITO PELO PREÇO mas não de maneira positiva
Canal: Carro Chefe
Data: 2025-05-23T20:56:04Z
URL da Thumbnail: https://i.ytimg.com/vi/r_FiBY59nf0/hqdefault.jpg
Link: https://www.youtube.com/watch?v=r_FiBY59nf0
Título: Avaliação Toyota Corolla 2003 - O MELHOR COROLLA JÁ FEITO! DESCCUBRA O SEGREDO PARA COMPRAR UM BOM
Canal: Carro Chefe
Data: 2025-02-12T20:56:55Z
URL da Thumbnail: https://i.ytimg.com/vi/JICFgcIv4TQ/hqdefault.jpg
Link: https://www.you

In [20]:
from googleapiclient.discovery import build
import os
from dotenv import load_dotenv

load_dotenv()
youtube = build('youtube', 'v3', developerKey=os.getenv('YOUTUBE_API_KEY'))

stats_response = youtube.videos().list(
            part='statistics',
            id='bAHpYCRMkTM'
        ).execute()

print(stats_response)

{'kind': 'youtube#videoListResponse', 'etag': '2WL7MYhvmzIKI3KG26yGj4m6j2I', 'items': [{'kind': 'youtube#video', 'etag': 'N-WZBXFUWirgXuvD6eXVUf2E72g', 'id': 'bAHpYCRMkTM', 'statistics': {'viewCount': '35595', 'likeCount': '3107', 'favoriteCount': '0', 'commentCount': '25'}}], 'pageInfo': {'totalResults': 1, 'resultsPerPage': 1}}


In [31]:

def definir_periodo_busca(meses_atras=6):
    # Data atual em UTC no formato RFC 3339
    agora = datetime.now(timezone.utc)
    
    # Data limite (agora menos X meses)
    data_inicio = agora - timedelta(days=meses_atras * 30)
    
    # Formatação para RFC 3339 (ex: 2026-04-21T12:00:00Z)
    # O 'Z' indica UTC, que é o padrão exigido pelo YouTube
    published_after = data_inicio.strftime('%Y-%m-%dT%H:%M:%SZ')
    published_before = agora.strftime('%Y-%m-%dT%H:%M:%SZ')
    
    return published_after, published_before

# Obter as datas
dt_ini, dt_fim = definir_periodo_busca(6)

order = 'relevance'  # ou 'date' para os mais recentes





query = f"review avaliação Ford Territory 2026"
search_response = youtube.search().list(
        q=query, type='video', maxResults=1 , part='snippet',     # Prioriza os mais relevantes
        relevanceLanguage='pt',
        publishedAfter=dt_ini,
        publishedBefore=dt_fim
    ).execute()

print(search_response)

{'kind': 'youtube#searchListResponse', 'etag': '6xNB7SCaKmUK-5NXNjJYPFc6RtA', 'nextPageToken': 'CAEQAA', 'regionCode': 'BR', 'pageInfo': {'totalResults': 1939, 'resultsPerPage': 1}, 'items': [{'kind': 'youtube#searchResult', 'etag': 'Dcu48ME1cwqJa30gQsWN3m1WaWM', 'id': {'kind': 'youtube#video', 'videoId': 'Hsb-Y8OqXaQ'}, 'snippet': {'publishedAt': '2026-04-02T23:09:38Z', 'channelId': 'UCV7ZpRaSA8EsuCNT3ouygWQ', 'title': 'Ford Territory 2026 - Super espaçoso e confortável mas NÃO É FLEX', 'description': 'Confira os links dos parceiros do Xenão com descontos e vantagens exclusivas pra você: Camisetas PaddockBR - Cupom ...', 'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/Hsb-Y8OqXaQ/default.jpg', 'width': 120, 'height': 90}, 'medium': {'url': 'https://i.ytimg.com/vi/Hsb-Y8OqXaQ/mqdefault.jpg', 'width': 320, 'height': 180}, 'high': {'url': 'https://i.ytimg.com/vi/Hsb-Y8OqXaQ/hqdefault.jpg', 'width': 480, 'height': 360}}, 'channelTitle': 'Carros do Xenão', 'liveBroadcastContent':

In [ ]:
from googleapiclient.discovery import build
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta, timezone

load_dotenv()
youtube = build('youtube', 'v3', developerKey=os.getenv('YOUTUBE_API_KEY'))


def definir_periodo_busca(meses_atras=6):
    # Data atual em UTC no formato RFC 3339
    agora = datetime.now(timezone.utc)
    
    # Data limite (agora menos X meses)
    data_inicio = agora - timedelta(days=meses_atras * 30)
    
    # Formatação para RFC 3339 (ex: 2026-04-21T12:00:00Z)
    # O 'Z' indica UTC, que é o padrão exigido pelo YouTube
    published_after = data_inicio.strftime('%Y-%m-%dT%H:%M:%SZ')
    published_before = agora.strftime('%Y-%m-%dT%H:%M:%SZ')
    
    return published_after, published_before

# Obter as datas
dt_ini, dt_fim = definir_periodo_busca(6)

order = 'relevance'  # ou 'date' para os mais recentes

def buscar_info_video(marca, modelo,ano,dt_ini,dt_fim,order='relevance'):
    query = f"review avaliação {marca} {modelo} {ano}"
    print(f"Buscando vídeos com a query: {query} entre {dt_ini} e {dt_fim} ordenados por {order}")
    # 1. Busca os vídeos
    search_response = youtube.search().list(
        q=query, part='snippet', type='video', maxResults=15 , order=order,         # Prioriza os mais relevantes
        relevanceLanguage='pt',
        publishedAfter=dt_ini,
        publishedBefore=dt_fim
    ).execute()

    resultados_finais = [] 

    for item in search_response['items']:
        video_id = item['id']['videoId']
        
        # 2. Busca estatísticas (incluindo views) usando o ID
        stats_response = youtube.videos().list(
            part='statistics',
            id=video_id
        ).execute()
        
        views = stats_response['items'][0]['statistics'].get('viewCount', '0')
        likes = stats_response['items'][0]['statistics'].get('likeCount', '0')
        favorites = stats_response['items'][0]['statistics'].get('favoriteCount', '0')
        comments = stats_response['items'][0]['statistics'].get('commentCount', '0')
        # 3. Organiza os dados
        video_info = {
            'titulo': item['snippet']['title'],
            'data': item['snippet']['publishedAt'],
            'link': f"https://www.youtube.com/watch?v={video_id}",
            'canal': item['snippet']['channelTitle'],
            'views': views,
            'likes': likes,
            'favorites': favorites,
            'comments': comments
        }
        resultados_finais.append(video_info)
        
    return resultados_finais

# Exemplo de execução
dados = buscar_info_video(marca="Ford", modelo="Territory", ano=2026, dt_ini=dt_ini, dt_fim=dt_fim, order=order)
for v in dados:
    print(f"Título: {v['titulo']}\nData: {v['data']}\nViews: {v['views']}\nLikes: {v['likes']}\nFavorites: {v['favorites']}\nComments: {v['comments']}\nLink: {v['link']}\nCanal: {v['canal']}\n")

Buscando vídeos com a query: review avaliação Ford Territory 2026 entre 2025-10-23T04:05:27Z e 2026-04-21T04:05:27Z ordenados por relevance
Título: Ford Territory 2026 - Super espaçoso e confortável mas NÃO É FLEX
Data: 2026-04-02T23:09:38Z
Views: 34816
Likes: 2419
Favorites: 0
Comments: 108
Link: https://www.youtube.com/watch?v=Hsb-Y8OqXaQ
Canal: Carros do Xenão

Título: AVALIAÇÃO NOVO FORD TERRITORY 2026 EM DETALHES
Data: 2026-04-04T17:16:52Z
Views: 3631
Likes: 218
Favorites: 0
Comments: 17
Link: https://www.youtube.com/watch?v=52hS7nvBKaE
Canal: Car Analysis | Rafael Amaral

Título: NOVO FORD TERRITORY 2026 Teste na vida real! Melhor que Renault BOREAL e TIGGO 8 PRO 2026?
Data: 2025-10-25T21:15:00Z
Views: 13618
Likes: 468
Favorites: 0
Comments: 47
Link: https://www.youtube.com/watch?v=3nKUA-FSM14
Canal: MyCars

Título: TEST DRIVE NOVO FORD TERRITORY TITANIUM 2026
Data: 2025-11-03T21:00:22Z
Views: 2307
Likes: 102
Favorites: 0
Comments: 6
Link: https://www.youtube.com/watch?v=O_I7KNX1

In [32]:
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta, timezone
from googleapiclient.discovery import build

# Configuração da API
load_dotenv()
youtube = build('youtube', 'v3', developerKey=os.getenv('YOUTUBE_API_KEY'))

def definir_periodo_busca(meses_atras=6):
    agora = datetime.now(timezone.utc)
    data_inicio = agora - timedelta(days=meses_atras * 30)
    return data_inicio.strftime('%Y-%m-%dT%H:%M:%SZ'), agora.strftime('%Y-%m-%dT%H:%M:%SZ')

def obter_dados_canal(channel_id):
    """Busca estatísticas detalhadas do canal."""
    response = youtube.channels().list(
        part='snippet,statistics',
        id=channel_id
    ).execute()
    
    if 'items' in response and response['items']:
        canal = response['items'][0]
        stats = canal['statistics']
        return {
            'nome': canal['snippet']['title'],
            'inscritos': stats.get('subscriberCount', '0'),
            'total_views_canal': stats.get('viewCount', '0'),
            'data_criacao': canal['snippet']['publishedAt']
        }
    return None

def buscar_info_video(marca, modelo, ano, dt_ini, dt_fim, order='relevance'):
    query = f"review avaliação {marca} {modelo} {ano}"
    print(f"--- Iniciando busca: {query} ---")
    
    # 1. Busca os vídeos
    search_response = youtube.search().list(
        q=query, part='snippet', type='video', maxResults=5, 
        order=order, relevanceLanguage='pt',
        publishedAfter=dt_ini, publishedBefore=dt_fim
    ).execute()

    resultados_finais = [] 

    for item in search_response['items']:
        video_id = item['id']['videoId']
        channel_id = item['snippet']['channelId']
        
        # 2. Busca estatísticas do vídeo
        stats_response = youtube.videos().list(
            part='statistics',
            id=video_id
        ).execute()
        
        stats = stats_response['items'][0]['statistics']
        
        # 3. Busca dados do canal
        canal_info = obter_dados_canal(channel_id)
        
        # 4. Organiza os dados
        video_info = {
            'titulo': item['snippet']['title'],
            'data': item['snippet']['publishedAt'],
            'link': f"https://www.youtube.com/watch?v={video_id}",
            'canal': item['snippet']['channelTitle'],
            'views': stats.get('viewCount', '0'),
            'likes': stats.get('likeCount', '0'),
            'favorites': stats.get('favoriteCount', '0'),
            'comments': stats.get('commentCount', '0'),
            'canal_inscritos': canal_info.get('inscritos') if canal_info else 'N/A'
        }
        resultados_finais.append(video_info)
        
    return resultados_finais

# --- Execução ---
dt_ini, dt_fim = definir_periodo_busca(6)
dados = buscar_info_video(marca="Ford", modelo="Territory", ano=2026, dt_ini=dt_ini, dt_fim=dt_fim)

for v in dados:
    print(f"Título: {v['titulo']}")
    print(f"Data: {v['data']} | Views: {v['views']} | Likes: {v['likes']}")
    print(f"Canal: {v['canal']} | Inscritos: {v['canal_inscritos']}")
    print(f"Link: {v['link']}\n" + "-"*50)

--- Iniciando busca: review avaliação Ford Territory 2026 ---
Título: Ford Territory 2026 - Super espaçoso e confortável mas NÃO É FLEX
Data: 2026-04-02T23:09:38Z | Views: 34818 | Likes: 2419
Canal: Carros do Xenão | Inscritos: 1170000
Link: https://www.youtube.com/watch?v=Hsb-Y8OqXaQ
--------------------------------------------------
Título: AVALIAÇÃO NOVO FORD TERRITORY 2026 EM DETALHES
Data: 2026-04-04T17:16:52Z | Views: 3631 | Likes: 218
Canal: Car Analysis | Rafael Amaral | Inscritos: 206000
Link: https://www.youtube.com/watch?v=52hS7nvBKaE
--------------------------------------------------
Título: NOVO FORD TERRITORY 2026 Teste na vida real! Melhor que Renault BOREAL e TIGGO 8 PRO 2026?
Data: 2025-10-25T21:15:00Z | Views: 13618 | Likes: 468
Canal: MyCars | Inscritos: 130000
Link: https://www.youtube.com/watch?v=3nKUA-FSM14
--------------------------------------------------
Título: TEST DRIVE NOVO FORD TERRITORY TITANIUM 2026
Data: 2025-11-03T21:00:22Z | Views: 2307 | Likes: 102
C

In [35]:
import os
import csv
from dotenv import load_dotenv
from datetime import datetime, timedelta, timezone
from googleapiclient.discovery import build

# Configuração inicial
load_dotenv()
API_KEY = os.getenv('YOUTUBE_API_KEY')

if not API_KEY:
    raise ValueError("A chave YOUTUBE_API_KEY não foi encontrada no arquivo .env")

youtube = build('youtube', 'v3', developerKey=API_KEY)

def definir_periodo_busca(meses_atras):
    """Retorna as datas de início e fim no formato RFC 3339 exigido pelo YouTube."""
    agora = datetime.now(timezone.utc)
    data_inicio = agora - timedelta(days=meses_atras * 30)
    return data_inicio.strftime('%Y-%m-%dT%H:%M:%SZ'), agora.strftime('%Y-%m-%dT%H:%M:%SZ')

def executar_pipeline_busca(marca, modelo, ano, meses_atras=6, max_resultados=15, order='relevance', arquivo_saida='resultados.csv'):
    """
    Função principal que gerencia a extração otimizada (Batching) e exportação para CSV.
    Todos os parâmetros de busca são dinâmicos e configuráveis na chamada.
    """
    dt_ini, dt_fim = definir_periodo_busca(meses_atras)
    query = f"review avaliação {marca} {modelo} {ano}"
    print(f"Buscando: '{query}' | Período: {dt_ini} até {dt_fim} | Max: {max_resultados}")
    
    # ---------------------------------------------------------
    # PASSO 1: Buscar os Vídeos (1 Requisição)
    # ---------------------------------------------------------
    search_response = youtube.search().list(
        q=query, part='snippet', type='video', maxResults=max_resultados, 
        order=order, relevanceLanguage='pt',
        publishedAfter=dt_ini, publishedBefore=dt_fim
    ).execute()

    itens_encontrados = search_response.get('items', [])
    if not itens_encontrados:
        print("Nenhum vídeo encontrado com esses parâmetros.")
        return

    # Extrai listas separadas de IDs para fazer as requisições em lote
    video_ids = [item['id']['videoId'] for item in itens_encontrados]
    # Usa 'set' para remover IDs de canais duplicados (caso um canal tenha 2 vídeos na lista)
    channel_ids = list(set([item['snippet']['channelId'] for item in itens_encontrados]))

    # ---------------------------------------------------------
    # PASSO 2: Buscar Estatísticas dos Vídeos em Lote (1 Requisição)
    # ---------------------------------------------------------
    # Passamos todos os IDs de uma vez separados por vírgula
    video_stats_response = youtube.videos().list(
        part='statistics', id=','.join(video_ids)
    ).execute()
    
    # Cria um dicionário de acesso rápido: { 'id_do_video': {estatisticas} }
    video_stats_map = {v['id']: v['statistics'] for v in video_stats_response.get('items', [])}

    # ---------------------------------------------------------
    # PASSO 3: Buscar Estatísticas dos Canais em Lote (1 Requisição)
    # ---------------------------------------------------------
    channel_stats_response = youtube.channels().list(
        part='statistics', id=','.join(channel_ids)
    ).execute()
    
    # Dicionário de acesso rápido: { 'id_do_canal': {estatisticas} }
    channel_stats_map = {c['id']: c['statistics'] for c in channel_stats_response.get('items', [])}

    # ---------------------------------------------------------
    # PASSO 4: Mesclar Dados e Preparar Base
    # ---------------------------------------------------------
    resultados_finais = []
    
    for item in itens_encontrados:
        v_id = item['id']['videoId']
        c_id = item['snippet']['channelId']
        
        # Resgata os dados dos mapas criados acima (ou dicionários vazios se houver falha)
        v_stats = video_stats_map.get(v_id, {})
        c_stats = channel_stats_map.get(c_id, {})
        
        video_info = {
            'video_id': v_id,
            'titulo': item['snippet']['title'],
            'data_publicacao': item['snippet']['publishedAt'],
            'canal_nome': item['snippet']['channelTitle'],
            'canal_id': c_id,
            'video_views': v_stats.get('viewCount', '0'),
            'video_likes': v_stats.get('likeCount', '0'),
            'video_comments': v_stats.get('commentCount', '0'),
            'canal_inscritos': c_stats.get('subscriberCount', '0'),
            'canal_total_views': c_stats.get('viewCount', '0'),
            'link_url': f"https://www.youtube.com/watch?v={v_id}"
        }
        resultados_finais.append(video_info)

    # ---------------------------------------------------------
    # PASSO 5: Salvar em CSV
    # ---------------------------------------------------------
    salvar_em_csv(resultados_finais, arquivo_saida)

def salvar_em_csv(dados, nome_arquivo):
    """Recebe a lista de dicionários e exporta para CSV."""
    if not dados:
        return
    
    # As chaves do primeiro dicionário serão os cabeçalhos do CSV
    cabecalhos = dados[0].keys()
    with open(nome_arquivo, mode='w', newline='', encoding='utf-8') as arquivo:
        writer = csv.DictWriter(arquivo, fieldnames=cabecalhos, delimiter=';')
        
        writer.writeheader()
        for linha in dados:
            writer.writerow(linha)
            
    print(f"\nSucesso! {len(dados)} registros salvos no arquivo '{nome_arquivo}'.")


# =========================================================
# EXECUÇÃO DO PIPELINE
# =========================================================
if __name__ == '__main__':
    # Você pode chamar essa função passando os dados do carro desejado
    executar_pipeline_busca(
        marca="Ford",
        modelo="Territory",
        ano=2026,
        meses_atras=10,          # Busca no último 1 ano
        max_resultados=20,       # Máximo de vídeos retornados (Limite: 50 por vez)
        order='viewCount',       # Traz os mais vistos primeiro (ou 'relevance', 'date')
        arquivo_saida='base_ford_territory.csv'
    )

Buscando: 'review avaliação Ford Territory 2026' | Período: 2025-06-25T04:40:37Z até 2026-04-21T04:40:37Z | Max: 20

Sucesso! 20 registros salvos no arquivo 'base_ford_territory.csv'.


In [36]:
import pandas as pd

df = pd.read_csv('base_ford_territory.csv', delimiter=';')
print(df.head())

      video_id                                             titulo  \
0  BkFNJYONHTM  A REVIRAVOLTA DA RENAULT! ATROPELOU COMPASS E ...   
1  w9p5i5NP0fA                     New CHERY Tiggo 8 2026! Review   
2  UQm_Awc4sLU  FORD TERRITORY 2025 (Titanium) - Test - Matías...   
3  GDNc13R1YA4                      La Ford Territory Es Distinta   
4  QDLRS1hiMcE  2026 Ford Territory Review UAE | Ford Territor...   

        data_publicacao     canal_nome                  canal_id  video_views  \
0  2025-11-06T19:00:22Z    Carro Chefe  UCMk1wBIsHmalYeE16GkD6bg       436191   
1  2025-10-16T00:01:13Z      NewCar TV  UCvD1NoWY8E0egM2uaUKcQTw       679543   
2  2025-08-31T23:12:13Z   MatiasAntico  UCd61IERUuGODiG08h3_30JA       362880   
3  2025-09-27T19:00:23Z  Lucas Abriata  UCOypnH-5DDOh1WXHJxm_t1A       298903   
4  2025-11-07T13:42:44Z     OTO Drive   UCt-TQyVs66iD0OifSIpnWDQ       297677   

   video_likes  video_comments  canal_inscritos  canal_total_views  \
0        31986            13

In [37]:
df

,video_id,titulo,data_publicacao,canal_nome,canal_id,video_views,video_likes,video_comments,canal_inscritos,canal_total_views,link_url
0,BkFNJYONHTM,A REVIRAVOLTA DA RENAULT! ATROPELOU COMPASS E ...,2025-11-06T19:00:22Z,Carro Chefe,UCMk1wBIsHmalYeE16GkD6bg,436191,31986,1317,1680000,537071690,https://www.youtube.com/watch?v=BkFNJYONHTM
1,w9p5i5NP0fA,New CHERY Tiggo 8 2026! Review,2025-10-16T00:01:13Z,NewCar TV,UCvD1NoWY8E0egM2uaUKcQTw,679543,8033,227,34900,30199721,https://www.youtube.com/watch?v=w9p5i5NP0fA
2,UQm_Awc4sLU,FORD TERRITORY 2025 (Titanium) - Test - Matías...,2025-08-31T23:12:13Z,MatiasAntico,UCd61IERUuGODiG08h3_30JA,362880,5424,553,697000,258029870,https://www.youtube.com/watch?v=UQm_Awc4sLU
3,GDNc13R1YA4,La Ford Territory Es Distinta,2025-09-27T19:00:23Z,Lucas Abriata,UCOypnH-5DDOh1WXHJxm_t1A,298903,12603,762,369000,55738085,https://www.youtube.com/watch?v=GDNc13R1YA4
4,QDLRS1hiMcE,2026 Ford Territory Review UAE | Ford Territor...,2025-11-07T13:42:44Z,OTO Drive,UCt-TQyVs66iD0OifSIpnWDQ,297677,500,43,1080000,6248989,https://www.youtube.com/watch?v=QDLRS1hiMcE
5,9v9iSnikNOw,Jetour T2 Review - The Best Value Adventure SU...,2025-12-17T13:00:21Z,Cars.co.za,UCB-mfYAd3oJLEkoMxjRAxbg,251671,2224,218,547000,196857864,https://www.youtube.com/watch?v=9v9iSnikNOw
6,0CzJO5FVnDE,Novo Renault BOREAL: Melhor SUV do Brasil? Det...,2025-11-06T19:00:01Z,Opinião Sincera,UCc0cU92Tf7rvd2gr-CpVO4Q,240880,10349,505,2150000,614129208,https://www.youtube.com/watch?v=0CzJO5FVnDE
7,y4_h7C-SBeY,Jetour T1 2026 Review,2025-07-30T22:01:09Z,AutoShow TV,UCVJdpj3jzdyq7QweTcUK9rg,377186,5814,24,458000,290931312,https://www.youtube.com/watch?v=y4_h7C-SBeY
8,MgFEBIYEO5I,Jetour T1 2026 Review Black Edition,2025-07-30T00:00:33Z,AutoShow TV,UCVJdpj3jzdyq7QweTcUK9rg,492047,4397,30,458000,290931312,https://www.youtube.com/watch?v=MgFEBIYEO5I
9,K5lvKm1H7kM,New CHERY Tiggo 9 2026! Review,2025-10-18T22:00:31Z,NewCar TV,UCvD1NoWY8E0egM2uaUKcQTw,414604,3071,26,34900,30199721,https://www.youtube.com/watch?v=K5lvKm1H7kM


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

def extrair_transcricao(video_id):
    """
    Busca a transcrição do vídeo mantendo a estrutura de objetos da API.
    Retorna o texto em um bloco único para uso em LLMs.
    """
    # 1. Cria a instância da API exatamente como no seu código
    youtube_api = YouTubeTranscriptApi()

    try:
        # 2. Lista as transcrições disponíveis usando o método da instância
        transcript_list = youtube_api.list(video_id)
        
        # 3. Encontra a transcrição (preferencialmente em português ou inglês)
        transcript = transcript_list.find_transcript(['pt', 'en'])
        
        # 4. Busca os dados
        texto_dados = transcript.fetch()
        
        # 5. Processa o resultado para gerar um texto contínuo
        textos_limpos = []
        for item in texto_dados:
            # Mantendo a sua verificação de dicionário vs objeto
            texto = item.get('text') if isinstance(item, dict) else item.text
            
            # Limpa quebras de linha e espaços extras do texto e guarda na lista
            if texto:
                textos_limpos.append(texto.replace('\n', ' ').strip())
        
        # Retorna o texto completo unido por espaços
        texto_completo = " ".join(textos_limpos)
        return texto_completo

    except Exception as e:
        # Avisa sobre o erro de forma silenciosa para não quebrar o pipeline
        print(f"  [Aviso] Transcrição indisponível para o vídeo {video_id}: {e}")
        return None

def limpar_texto_para_llm(texto):
    """
    Limpa o texto removendo ruídos para otimizar o consumo de tokens 
    e melhorar o entendimento do LLM.
    """
    if not isinstance(texto, str) or not texto.strip():
        return ""
    
    # 1. Remove URLs e Links (O LLM não consegue clicar, só gastaria tokens)
    texto = re.sub(r'http\S+|www\.\S+', '', texto)
    
    # 2. Remove tags HTML residuais (muito comum em comentários do YouTube)
    texto = re.sub(r'<.*?>', '', texto)
    
    # 3. Substitui quebras de linha (\n), tabulações (\t) e múltiplos espaços por um único espaço
    texto = re.sub(r'\s+', ' ', texto)
    
    # 4. Remove repetições exageradas de pontuação (ex: "carro ruim!!!!!" vira "carro ruim!")
    # Isso evita que o modelo dê peso excessivo a um grito em vez do contexto
    texto = re.sub(r'([.!?])\1+', r'\1', texto)
    
    # 5. Remove aspas duplas/simples irregulares que podem quebrar a formatação JSON na saída do LLM
    texto = texto.replace('"', "'").replace('`', "'")
    
    # 6. Limpa espaços no início e no fim
    return texto.strip()

# --- Testando a função ---
if __name__ == "__main__":
    vid = 'BmYsInLDmrI'
    resultado = extrair_transcricao(vid)
    
    if resultado:
        print(f"Transcrição extraída com sucesso! ({len(resultado)} caracteres)")
        print(f"Trecho: {resultado[:200]}...")

        print(resultado)

Transcrição extraída com sucesso! (17405 caracteres)
Trecho: Esse é o novo Ford Territory 2025. Mudou tudo, agora anda de verdade. É um dos maiores espaços internos da categoria. Mas será que tem preço e condições para bater de frente com os outros chineses? Nã...
Esse é o novo Ford Territory 2025. Mudou tudo, agora anda de verdade. É um dos maiores espaços internos da categoria. Mas será que tem preço e condições para bater de frente com os outros chineses? Não, você não me escutou errado, não. Esse, apesar de ter o emblema da Ford, vem da China. O SUV de entrada agora da Ford no Brasil, mas que não é feito pela Ford. E além disso, vale a pena dizer que esse carro lá fora já ganhou o facelift e virou híbrido. Aqui também vai ser igual no vídeo de hoje a gente vai ver tudo sobre essa máquina aqui. Vamos falar da motorização, vamos falar do câmbio do espaço interno que ele dá um banho nos outros da categoria do porta presuntos. E é claro que a gente vai levar essa máquina, dar um rolê e

In [ ]:
import os
from googleapiclient.discovery import build
from dotenv import load_dotenv

# Reutilizando a função de limpeza definida anteriormente
import re

def limpar_texto_para_llm(texto):
    if not isinstance(texto, str) or not texto.strip(): return ""
    texto = re.sub(r'http\S+|www\.\S+', '', texto)
    texto = re.sub(r'<.*?>', '', texto)
    texto = re.sub(r'\s+', ' ', texto)
    texto = re.sub(r'([.!?])\1+', r'\1', texto)
    return texto.replace('"', "'").strip()

def coletar_comentarios(video_id, max_resultados=200):
    """
    Coleta comentários de um vídeo usando paginação.
    Retorna uma lista de dicionários com autor, texto limpo e métricas.
    """
    load_dotenv()
    youtube = build('youtube', 'v3', developerKey=os.getenv('YOUTUBE_API_KEY'))
    
    comentarios_totais = []
    next_page_token = None
    
    try:
        while len(comentarios_totais) < max_resultados:
            # Executa a requisição
            response = youtube.commentThreads().list(
                part='snippet',
                videoId=video_id,
                maxResults=min(100, max_resultados - len(comentarios_totais)),
                textFormat='plainText',
                order='relevance',
                pageToken=next_page_token
            ).execute()
            
            for item in response.get('items', []):
                snippet = item['snippet']['topLevelComment']['snippet']
                
                texto_bruto = snippet['textDisplay']
                texto_limpo = limpar_texto_para_llm(texto_bruto)
                
                # Só adicionamos se o comentário não ficou vazio após a limpeza
                if texto_limpo:
                    comentarios_totais.append({
                        'texto': texto_limpo,
                        'likes': snippet['likeCount'],
                        'total_respostas': item['snippet']['totalReplyCount'],
                        'data': snippet['publishedAt']
                    })
            
            # Verifica se existe uma próxima página
            next_page_token = response.get('nextPageToken')
            if not next_page_token:
                break
                
        print(f"  [Sucesso] {len(comentarios_totais)} comentários coletados para o vídeo {video_id}")
        return comentarios_totais

    except Exception as e:
        print(f"  [Erro] Falha ao coletar comentários do vídeo {video_id}: {e}")
        return []

# --- Exemplo de Uso ---
if __name__ == "__main__":
    v_id = 'BmYsInLDmrI'
    lista_comentarios = coletar_comentarios(v_id, max_resultados=50)

  [Sucesso] 50 comentários coletados para o vídeo BmYsInLDmrI
[@PauloSergio-tz7sj]: Só lembro da piada 'mais rápido que o Ford territory 'kkkkkkk... (Likes: 131)
[@Marquiiiz]: Estou com ele a 6 meses, tem sido a melhor escolha pra mim, carrro super confortavel, ate aqui nenhu... (Likes: 93)
[@nicolasborges2022]: Inacreditável o acabamento, o conforto e o espaço deste carro! 👏🏻 Muito bem completo e com vários it... (Likes: 44)


In [46]:
import os
import re
import time
import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build
from youtube_transcript_api import YouTubeTranscriptApi
# ==========================================
# 3. EXECUÇÃO DO PIPELINE
# ==========================================
if __name__ == "__main__":
    load_dotenv()
    
    # Simulando a carga da sua base de dados (Poderia ser pd.read_csv('seus_videos.csv'))

    df_videos = pd.read_csv('base_ford_territory.csv', delimiter=';')
    
    linhas_transcricao = []
    linhas_comentarios = []
    
    print("Iniciando processamento dos vídeos...\n")
    
    for index, row in df_videos.iterrows():
        vid_id = row['video_id']
        canal = row['canal_nome']
        titulo = row['titulo']
        
        print(f"Processando [{index+1}/{len(df_videos)}]: {titulo[:40]}...")
        
        # --- ETAPA A: TRANSCRIÇÃO ---
        print("  -> Baixando transcrição...")
        texto_transcricao = extrair_transcricao(vid_id)
        
        linhas_transcricao.append({
            'video_id': vid_id,
            'canal_nome': canal,
            'titulo_video': titulo,
            'texto_transcricao': texto_transcricao,
            'llm_categoria_veiculo': None,
            'llm_pontos_positivos': None,
            'llm_pontos_negativos': None,
            'llm_nota_geral': None
        })
        
        # --- ETAPA B: COMENTÁRIOS ---
        print("  -> Coletando comentários do público...")
        comentarios_extraidos = coletar_comentarios(vid_id, max_resultados=50) # Limitado a 50 para teste rápido
        
        for c in comentarios_extraidos:
            linhas_comentarios.append({
                'video_id': vid_id,
                'canal_nome': canal,
                'texto_comentario': c['texto'],
                'likes_comentario': c['likes'],
                'data_comentario': c['data'],
                'llm_sentimento': None,
                'llm_tema_principal': None
            })
            
        # Pequena pausa para evitar bloqueios por taxa de requisição (Rate Limit)
        time.sleep(1)

    # ==========================================
    # 4. SALVANDO OS DADOS PROCESSADOS
    # ==========================================
    df_transcricoes = pd.DataFrame(linhas_transcricao)
    df_comentarios = pd.DataFrame(linhas_comentarios)
    
    # Salvando em CSV
    df_transcricoes.to_csv('pipeline_transcricoes_real.csv', index=False, sep=';', encoding='utf-8')
    df_comentarios.to_csv('pipeline_comentarios_real.csv', index=False, sep=';', encoding='utf-8')
    
    print("\n✅ Processamento concluído com sucesso!")
    print(f"Total de Transcrições processadas: {len(df_transcricoes)}")
    print(f"Total de Comentários coletados: {len(df_comentarios)}")
    print("Arquivos salvos: 'pipeline_transcricoes_real.csv' e 'pipeline_comentarios_real.csv'")

Iniciando processamento dos vídeos...

Processando [1/20]: A REVIRAVOLTA DA RENAULT! ATROPELOU COMP...
  -> Baixando transcrição...
  -> Coletando comentários do público...
  [Sucesso] 50 comentários coletados para o vídeo BkFNJYONHTM
Processando [2/20]: New CHERY Tiggo 8 2026! Review...
  -> Baixando transcrição...
  [Aviso] Transcrição indisponível para o vídeo w9p5i5NP0fA: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=w9p5i5NP0fA! This is most likely caused by:

Subtitles are disabled for this video

If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide the information needed to replicate the error. Also make sure that there are no open issues which already describe your problem!
  -> Coletando comentários do público...
  [Suce

In [47]:
df_comentarios = pd.read_csv('pipeline_comentarios_real.csv', delimiter=';')
df_comentarios

,video_id,canal_nome,texto_comentario,likes_comentario,data_comentario,llm_sentimento,llm_tema_principal
0,BkFNJYONHTM,Carro Chefe,Cadê o tapa no like? 😂,613,2025-11-06T19:02:46Z,NaN,NaN
1,BkFNJYONHTM,Carro Chefe,"Salve Lucão, sou um dos analistas de engenhari...",2713,2025-11-06T21:56:35Z,NaN,NaN
2,BkFNJYONHTM,Carro Chefe,27kg de torque Corrente de comando 4 cilindros...,813,2025-11-07T17:30:08Z,NaN,NaN
3,BkFNJYONHTM,Carro Chefe,"por essa ninguém esperava, finalmente uma mont...",401,2025-11-06T21:24:03Z,NaN,NaN
4,BkFNJYONHTM,Carro Chefe,Ficou bonito. E o acabamento faz o taos chorar...,572,2025-11-06T19:59:30Z,NaN,NaN
...,...,...,...,...,...,...,...
800,J4OQCaOpQgA,AutoShow TV,ai,0,2026-04-16T07:15:29Z,NaN,NaN
801,J4OQCaOpQgA,AutoShow TV,CarPlay doesn’t work while driving,0,2026-04-02T17:26:21Z,NaN,NaN
802,J4OQCaOpQgA,AutoShow TV,I love mine. Glad I got rid of my Tesla,0,2026-01-04T00:57:54Z,NaN,NaN
803,J4OQCaOpQgA,AutoShow TV,గోడి బాయ్ ok,2,2025-07-14T12:12:02Z,NaN,NaN


In [48]:
df_transcricoes = pd.read_csv('pipeline_transcricoes_real.csv', delimiter=';')
df_transcricoes

,video_id,canal_nome,titulo_video,texto_transcricao,llm_categoria_veiculo,llm_pontos_positivos,llm_pontos_negativos,llm_nota_geral
0,BkFNJYONHTM,Carro Chefe,A REVIRAVOLTA DA RENAULT! ATROPELOU COMPASS E ...,Esse é o novo Renault Boreal e é o melhor carr...,NaN,NaN,NaN,NaN
1,w9p5i5NP0fA,NewCar TV,New CHERY Tiggo 8 2026! Review,NaN,NaN,NaN,NaN,NaN
2,UQm_Awc4sLU,MatiasAntico,FORD TERRITORY 2025 (Titanium) - Test - Matías...,NaN,NaN,NaN,NaN,NaN
3,GDNc13R1YA4,Lucas Abriata,La Ford Territory Es Distinta,NaN,NaN,NaN,NaN,NaN
4,QDLRS1hiMcE,OTO Drive,2026 Ford Territory Review UAE | Ford Territor...,today. We have the Ford territory with us. Now...,NaN,NaN,NaN,NaN
5,9v9iSnikNOw,Cars.co.za,Jetour T2 Review - The Best Value Adventure SU...,[music] [music] Hello and welcome back to the ...,NaN,NaN,NaN,NaN
6,0CzJO5FVnDE,Opinião Sincera,Novo Renault BOREAL: Melhor SUV do Brasil? Det...,"Este é o novíssimo Renault Boreal, USV grande,...",NaN,NaN,NaN,NaN
7,y4_h7C-SBeY,AutoShow TV,Jetour T1 2026 Review,NaN,NaN,NaN,NaN,NaN
8,MgFEBIYEO5I,AutoShow TV,Jetour T1 2026 Review Black Edition,NaN,NaN,NaN,NaN,NaN
9,K5lvKm1H7kM,NewCar TV,New CHERY Tiggo 9 2026! Review,NaN,NaN,NaN,NaN,NaN
